# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset and view basic metadata
dataset = mlc.Dataset(croissant_url)

print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Version: {dataset.metadata.version}")

# Optionally view keywords and license
print(f"Keywords: {getattr(dataset.metadata, 'keywords', [])}")
print(f"License: {dataset.metadata.license}")


## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.


In [ ]:
# List available record sets
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record set name: {rs.name}, @id: {rs.id}")

# For each record set, display its fields and their @id values
for rs in record_sets:
    print(f"\n---\nFields in record set '{rs.name}' (@id: {rs.id}):")
    for field in rs.fields:
        print(f"\tField name: {field.name}  |  @id: {field.id}  |  DataType: {getattr(field, 'data_type', None)}")


## 3. Data Extraction
Load records from each record set into a DataFrame for analysis. You can select specific record set and field `@id`s you want to work with, and reference them using the variables below.

In [ ]:
# Extract data from record sets by their @id

# Choose record set ids to extract - here we list all record set @id values found
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

# Load each as a DataFrame
for record_set_id in record_set_ids:
    df = pd.DataFrame(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for record set @id: {record_set_id}  |  {df.shape[0]} records  x  {df.shape[1]} fields")

# Show columns of the main record set for demonstration
main_record_set_id = record_set_ids[0]  # typically your main data table
print(f"\nColumns in main record set (@id: {main_record_set_id}): \n{dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Now that your data is loaded, you can process and analyze it. In this example we'll:
- Filter records by a numeric field (e.g., 'phq9_score', if available in your fields),
- Normalize the values,
- Group the result by a categorical field (e.g., by 'village_name' or 'gender').

Be sure to use field and group field `@id` values for referencing columns.

In [ ]:
# Choose a numeric field @id and (optionally) a group field @id
# For demonstration, we will attempt a few likely field names (replace as needed)

main_df = dataframes[main_record_set_id]

possible_numeric_fields = ['phq9_score', 'gad7_score', 'psq_score', 'age']
numeric_field = None
for field in possible_numeric_fields:
    if field in main_df.columns:
        numeric_field = field
        break
if numeric_field is None:
    raise ValueError("No recognized numeric field found in the main record set DataFrame.")

# Define a threshold value for filtering
threshold = 10

filtered_df = main_df[main_df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field (z-score normalization)
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a demographic field if it exists
possible_group_fields = ['gender', 'village_name', 'marital_status', 'level_of_education']
group_field = None
for col in possible_group_fields:
    if col in filtered_df.columns:
        group_field = col
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nMean {numeric_field} grouped by {group_field}:")
    print(grouped_df.head())
else:
    print("\nNo suitable group field found for grouping.")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we'll plot the distribution of the selected numeric field, and (optionally) compare groups if available.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution histogram for the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(main_df[numeric_field].dropna(), bins=20, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If group_field exists, show boxplot by group
if group_field:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=main_df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()


## 6. Conclusion
In this notebook, we demonstrated how to use `mlcroissant` to load and explore the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya. 

- We reviewed the metadata and available record sets and fields (referenced by `@id`).
- We loaded main survey data, filtered and normalized a mental health indicator (e.g., PHQ-9, GAD-7, or PSQ scores), and explored group differences by demographics.
- We've visualized data distributions and group comparisons.

This structured workflow allows for reliable, reproducible data exploration aligned with FAIR and Croissant principles. For more advanced usage—including joining datasets, customizing workflows, or exporting derived data—refer to the [mlcroissant documentation](https://github.com/mlcommons/croissant) and the FAIR² dataset's own documentation.
